In [1]:
%%writefile helper.py
import random
import time

def hitung_prima(n):
    """Menghitung jumlah bilangan prima dari 2 hingga n-1"""
    count = 0
    for i in range(2, n):
        prime = True
        for j in range(2, int(i**0.5) + 1):
            if i % j == 0:
                prime = False
                break
        if prime:
            count += 1
    return count

def worker_task(num_angka):
    """Menghitung subtotal angka acak untuk pemrosesan paralel"""
    return sum(random.randint(1, 100) for _ in range(num_angka))


def producer(q, stop_event):
    """Menghasilkan beban kerja dan memasukkannya ke Queue"""
    for i in range(1, 4):
        item = 50000 + (i * 20000)
        q.put(item)
        time.sleep(0.1)
    stop_event.set()

def consumer(q, stop_event):
    """Mengambil tugas dari Queue dan memproses perhitungan prima"""
    while not (stop_event.is_set() and q.empty()):
        try:
            item = q.get(timeout=0.1)
            count = hitung_prima(item)
        except:
            continue

Overwriting helper.py


In [4]:
#1
import time
import multiprocessing as mp
from helper import hitung_prima

if __name__ == '__main__':
    angka_uji = 100000  
    workers_list = [1, 2, 4, 8]
    waktu_eksekusi = {}
    speedup_aktual = []
    speedup_teori = [1.00, 1.74, 2.76, 3.90]

    t1 = 0
    print("=== [NO. 1] ANALISIS SPEEDUP SKALA ===")
    print(f"{'Worker (N)':^10} | {'Waktu (s)':^12} | {'Speedup Aktual':^15} | {'Speedup Teoretis (Amdahl P=0.85)':^32}")
    print("-" * 75)

    for idx, w in enumerate(workers_list):
        start_time = time.time()
        
        with mp.Pool(processes=w) as pool:
            pool.map(hitung_prima, [angka_uji // w] * w)
        
        elapsed = time.time() - start_time
        waktu_eksekusi[w] = elapsed
        if w == 1:
            t1 = elapsed
        
        speedup = t1 / elapsed
        speedup_aktual.append(speedup)
        print(f"{w:^10} | {elapsed:^12.4f} | {speedup:^14.2f}x | {speedup_teori[idx]:^32.2f}x")

=== [NO. 1] ANALISIS SPEEDUP SKALA ===
Worker (N) |  Waktu (s)   | Speedup Aktual  | Speedup Teoretis (Amdahl P=0.85)
---------------------------------------------------------------------------
    1      |    0.6657    |      1.00     x |               1.00              x
    2      |    0.4210    |      1.58     x |               1.74              x
    4      |    0.3866    |      1.72     x |               2.76              x
    8      |    0.4007    |      1.66     x |               3.90              x


In [ ]:
penjelasan 1
Sesuai data teoretis Hukum Amdahl dengan fraksi paralel P = 0.85, speedup teoretis berturut-turut adalah 1.00x (N=1), 1.74x (N=2), 2.76x $N=4), dan 3.90x (N=8), dengan batas maksimum limiting speedup sebesar 6.67x.
Penyebab Deviasi Kurva Aktual: Keterbatasan Core Fisik: Saat jumlah worker ($N=8$) melebihi jumlah core fisik CPU komputer, terjadi context switching yang memicu tambahan beban (overhead) waktu.
Overhead Pembuatan Proses: Proses forking memakan waktu untuk alokasi memori dan sinkronisasi yang tidak diperhitungkan dalam rumus Amdahl yang ideal.

In [1]:
%%writefile helper_no2.py
import random

def worker_task(num_angka):
    return sum(random.randint(1, 100) for _ in range(num_angka))

Writing helper_no2.py


In [3]:
import time
from concurrent.futures import ProcessPoolExecutor
from helper_no2 import worker_task

if __name__ == '__main__':
    total_angka = 4000000
    pembagian = total_angka // 4
    
    start_time = time.time()
    
    with ProcessPoolExecutor(max_workers=4) as executor:
        hasil_list = list(executor.map(worker_task, [pembagian] * 4))
        
    total_hasil = sum(hasil_list)
    elapsed = time.time() - start_time
    
    print(f"Hasil Total: {total_hasil}")
    print(f"Waktu Eksekusi: {elapsed:.4f} detik")

Hasil Total: 201927114
Waktu Eksekusi: 1.3160 detik


In [ ]:
penjelasan 2
Proses-proses yang berjalan secara terpisah saling mengirimkan data atau hasil sub-komputasi menggunakan struktur perantara (seperti Queue atau pengumpulan hasil langsung dari Pool). Proses-proses yang berjalan secara paralel mengakses dan memodifikasi satu variabel atau ruang memori yang sama secara langsung (menggunakan Value dan Lock agar aman dari kondisi balap/race condition).
helper_no2.py: Diperlukan dalam lingkungan Windows karena metode spawn pada multiprocessing mengharuskan fungsi worker didefinisikan dalam modul terpisah agar dapat di-serialize dengan benar ke setiap proses anak.

In [2]:
#3
import time
import multiprocessing as mp
from helper import producer, consumer

if __name__ == '__main__':
    q = mp.Queue()
    stop_event = mp.Event()
    
    p_prod = mp.Process(target=producer, args=(q, stop_event))
    p_cons = mp.Process(target=consumer, args=(q, stop_event))
    
    start_time = time.time()
    p_prod.start()
    p_cons.start()
    
    p_prod.join()
    p_cons.join()
    
    elapsed = time.time() - start_time

In [ ]:
penjelasan 3
mplementasi pemrograman paralel pada Jupyter Notebook di Windows memerlukan pemisahan fungsi ke dalam modul eksternal seperti file helper.py guna mengatasi kendala serialisasi dan metode spawn. Nomor 1 dan 2 menggunakan pendekatan Message Passing serta ProcessPoolExecutor untuk membagi beban kerja komputasi berat—seperti pencarian bilangan prima dan penjumlahan acak—ke dalam beberapa proses paralel secara efisien. Selanjutnya, Nomor 3 menerapkan pola Producer-Consumer menggunakan mp.Queue dan mp.Event, di mana proses produsen bertugas menghasilkan beban kerja dan memasukkannya ke dalam antrean, sementara proses konsumen secara bersamaan mengambil serta memproses data tersebut hingga tuntas tanpa mengalami kemacetan eksekusi.

In [ ]:
4. refleksi singkat
Manager() dipilih saat aplikasi memerlukan struktur data kompleks yang dikelola bersama antar-proses—seperti list, dict, atau custom object—tanpa batasan tipe data primitif, karena menyediakan proksi berbasis server yang sangat mudah digunakan (ease of use). Namun, pendekatan ini memiliki trade-off performa yang lebih lambat akibat overhead komunikasi antar-proses yang intensif.
Sebaliknya, Value dan Array dipilih ketika prioritas utama adalah performa tinggi dan latensi rendah untuk data numerik primitif (seperti integer atau float) yang disimpan langsung di memori bersama (shared memory) tanpa perantara server. Konsekuensinya, penggunaannya lebih kaku, terbatas pada tipe data dasar, dan memerlukan mekanisme manual seperti Lock untuk menghindari race condition.